# Survivorship check (Task A) — READ-ONLY

How much survivorship tilt does the Bukowski corpus carry? Two reads, nothing selected, nothing logged as a trial.

- **A1** file end dates: how many tickers stop before 2024-01-01 (dead names kept) vs total, and live/born/died
  counts per year. Lockbox: series are cut at 2024-12-31 at load, so a "last date" is never later than the cut.
  Names whose (cut) last bar is the final bar of the sample are reported as a COUNT only.
- **A2** equal-weight, daily-rebalanced buy-and-hold of the full universe, 2011-01-03 .. 2023-12-29 (buy at the
  2010-12-31 close), vs RSP / IWM / IJR / SPY total return (all series split+dividend adjusted). Repeated for the
  bottom tercile of 2011 median dollar volume (fixed cohort). CAGR, max drawdown, per-calendar-year gap.

**Rules (pre-declared):** a ticker contributes its close-to-close return on day t iff it has valid closes on t and its
previous bar. Bad prints = daily return > +300% or < −95%: counted, listed, and ONLY that ticker-day return is
dropped (the ticker stays). Results are shown with and without bad prints; the clean series is primary.

**Caveats:** no delisting returns exist in the corpus, so a dying name's final collapse is missing — the gap is a
LOWER bound on the bias. Daily rebalancing harvests a rebalancing/small-cap premium that quarterly-rebalanced RSP
does not; part of any gap vs RSP is that, not survivorship. Tercile cohort uses full-2011 dollar volume (in-year
look-ahead; irrelevant for a survivorship read).

In [ ]:
# ===== Cell 0 — config =====
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass                                            # local smoke test: BUKOWSKI_DIR points at fixtures
import pandas as pd, numpy as np, glob, json, os, math, urllib.request
from pathlib import Path
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path(os.environ.get('BUKOWSKI_DIR', '/content/drive/MyDrive/Bukowski'))
REF_DIR  = DATA_DIR/'reference'
OUT_DIR  = DATA_DIR/'results'/'screens'; OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- LOCKBOX ----
CUT_DATE = pd.Timestamp('2024-12-31')     # every series cut at load; nothing after this is read, pulled or printed

A_START, A_END = pd.Timestamp('2011-01-01'), pd.Timestamp('2023-12-31')
EARLY_END      = pd.Timestamp('2024-01-01')
BAD_UP, BAD_DN = 3.0, -0.95                   # daily return bad-print thresholds
BENCH          = ['RSP', 'IWM', 'IJR', 'SPY']  # SPY = corpus DATA_DIR/SPY.csv; the rest from reference/
# Funds in DATA_DIR/*.csv are not stocks: excluded from the universe (the ones actually found are printed).
ETF_EXCLUDE = {'SPY', 'RSP', 'IWM', 'IJR', 'QQQ', 'DIA', 'MDY', 'IWB', 'IWV', 'IWN', 'IWO', 'IWD', 'IWF', 'VTI',
               'VOO', 'IVV', 'VB', 'VO', 'XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV',
               'XLY', 'GLD', 'SLV', 'TLT', 'IEF', 'SHY', 'HYG', 'LQD', 'AGG', 'BND', 'EEM', 'EFA', 'VNQ', 'SMH',
               'SOXX', 'XBI', 'IBB', 'KRE', 'KBE', 'XHB', 'XRT', 'XME', 'XOP', 'ARKK', 'GDX', 'GDXJ', 'USO',
               'UNG', 'UUP', 'VXX', 'TQQQ', 'SQQQ', 'SPXL', 'SPXS', 'UVXY', 'SSO', 'SDS'}

In [ ]:
# ===== Cell 1 — helpers + benchmark data (pull IJR/IWM/RSP if missing) =====
def load_px(path):
    """Read OHLC(+V) and CUT at CUT_DATE before anything else touches it."""
    cols = pd.read_csv(path, nrows=0).columns
    use = [c for c in ['Date', 'Open', 'High', 'Low', 'Close', 'Volume'] if c in cols]
    d = pd.read_csv(path, usecols=use); d['Date'] = pd.to_datetime(d['Date'])
    d = d.sort_values('Date').drop_duplicates('Date')
    d = d[d['Date'] <= CUT_DATE].reset_index(drop=True)                  # LOCKBOX
    if 'Volume' not in d.columns: d['Volume'] = np.nan
    return d


def tiingo_token():
    token = os.environ.get('TIINGO_API_KEY')
    if not token:
        try:
            from google.colab import userdata; token = userdata.get('TIINGO_API_KEY')
        except Exception:
            token = None
    return token


def ref_path(tk):
    """RSP historically lives at DATA_DIR/RSP.csv or REF_DIR/RSP.csv; everything new goes to REF_DIR."""
    return DATA_DIR/f'{tk}.csv' if tk == 'RSP' and (DATA_DIR/'RSP.csv').exists() else REF_DIR/f'{tk}.csv'


def ensure_ref(tk):
    """One-off ADJUSTED Tiingo pull, only if missing. endDate = CUT_DATE so sealed rows never reach Drive."""
    path = ref_path(tk)
    if path.exists():
        print(f'{tk}: present at {path} — no pull'); return path
    token = tiingo_token()
    assert token, f'Set TIINGO_API_KEY (Colab secret or env) to pull {tk}.'
    url = (f'https://api.tiingo.com/tiingo/daily/{tk}/prices?startDate=2003-01-01'
           f'&endDate={CUT_DATE:%Y-%m-%d}&token={token}')
    js = pd.DataFrame(json.load(urllib.request.urlopen(url, timeout=60)))
    df = pd.DataFrame({'Date': pd.to_datetime(js['date']).dt.strftime('%Y-%m-%d'),
                       'Open': js['adjOpen'], 'High': js['adjHigh'], 'Low': js['adjLow'],
                       'Close': js['adjClose'], 'Volume': js['adjVolume']})
    df = df[pd.to_datetime(df['Date']) <= CUT_DATE]                     # belt and braces
    REF_DIR.mkdir(parents=True, exist_ok=True); df.to_csv(path, index=False)
    print(f'{tk}: pulled {len(df)} rows (adjusted) -> {path}')
    return path


def close_ret(d, cal=None):
    """Close-to-close simple return on the series' own rows (optionally restricted to a calendar first)."""
    s = d.set_index('Date')['Close'].astype(float)
    if cal is not None: s = s[s.index.isin(cal)]
    s = s.where(s > 0)
    return (s / s.shift(1) - 1).dropna()

BPX = {'SPY': load_px(DATA_DIR/'SPY.csv')}
for tk in ['RSP', 'IWM', 'IJR']:
    BPX[tk] = load_px(ensure_ref(tk))
CAL = pd.DatetimeIndex(BPX['SPY']['Date'])               # trading calendar = SPY rows
assert all(d['Date'].max() <= CUT_DATE for d in BPX.values())

In [ ]:
# ===== Cell 2 — load the universe (cut at load) =====
files = sorted(glob.glob(f'{DATA_DIR}/*.csv'))
etfs_found = sorted(Path(f).stem.upper() for f in files if Path(f).stem.upper() in ETF_EXCLUDE)
RET, DVOL, FIRST, LAST, NBAR, EMPTY, FAILED = {}, {}, {}, {}, {}, [], []
for f in files:
    tk = Path(f).stem.upper()
    if tk in ETF_EXCLUDE: continue
    try:
        d = load_px(f)
    except Exception as e:
        FAILED.append((tk, repr(e)[:80])); continue
    if d.empty:
        EMPTY.append(tk); continue                        # no rows on/before the cut
    FIRST[tk], LAST[tk], NBAR[tk] = d['Date'].iloc[0], d['Date'].iloc[-1], len(d)
    RET[tk] = close_ret(d, CAL)
    y11 = d[d['Date'].dt.year == 2011]
    DVOL[tk] = (len(y11), float((y11['Close'] * y11['Volume']).median()) if len(y11) else np.nan)
assert max(LAST.values()) <= CUT_DATE
print(f'files {len(files)} | ETFs excluded {len(etfs_found)}: {etfs_found}')
print(f'stock universe {len(RET)} | no rows <= cut: {len(EMPTY)} | unreadable: {len(FAILED)} {FAILED[:5]}')

## A1 — file end dates

In [ ]:
# ===== Cell 3 — A1: who ends early =====
LAST_BAR = CAL[-1]                                       # final bar of the (cut) sample
ends = pd.DataFrame({'first': pd.Series(FIRST), 'last': pd.Series(LAST), 'bars': pd.Series(NBAR)})
early   = ends[ends['last'] <  EARLY_END].sort_values('last')
in_2024 = ends[(ends['last'] >= EARLY_END) & (ends['last'] < LAST_BAR)].sort_values('last')
n_tot = len(ends)
print(f'A1: {len(early)} / {n_tot} tickers ({len(early)/n_tot:.1%}) end before {EARLY_END:%Y-%m-%d}')
print(f'    {len(in_2024)} end during 2024 before the final bar; '
      f'{n_tot - len(early) - len(in_2024)} run to the final in-sample bar (count only)')
pd.set_option('display.max_rows', 500)
print('\nEnding before 2024-01-01 (last date = last bar):')
print(early.assign(first=early['first'].dt.date, last=early['last'].dt.date).to_string())
if len(in_2024):
    print('\nEnding during 2024 (before the final bar):')
    print(in_2024.assign(first=in_2024['first'].dt.date, last=in_2024['last'].dt.date).to_string())
early.to_csv(OUT_DIR/'survivorship_a1_early_files.csv')

# live / born / died per year (2010 "born" = data start, not an IPO)
yrs = range(2010, 2025)
fy, ly = ends['first'].dt.year, ends['last'].dt.year
per_year = pd.DataFrame({'live': [int(((fy <= y) & (ly >= y)).sum()) for y in yrs],
                         'born': [int((fy == y).sum()) for y in yrs],
                         'died': [int(((ly == y) & (ends['last'] < LAST_BAR)).sum()) for y in yrs]},
                        index=list(yrs))
print('\nLive / born / died per year (died = last bar in that year and before the final bar):')
print(per_year.to_string())

## A2 — equal-weight buy-and-hold vs RSP / IWM / IJR / SPY

In [ ]:
# ===== Cell 4 — bad prints + EW daily series =====
R = pd.DataFrame(RET).reindex(CAL)                      # day x ticker, NaN = not held that day
bad = (R > BAD_UP) | (R < BAD_DN)
bp = R[bad].stack().dropna().rename('ret').reset_index(); bp.columns = ['date', 'ticker', 'ret']
bp = bp.sort_values(['date', 'ticker']).reset_index(drop=True)
in_win = bp[(bp['date'] >= A_START) & (bp['date'] <= A_END)]
print(f'bad prints (>{BAD_UP:+.0%} or <{BAD_DN:+.0%}): {len(bp)} total, {len(in_win)} inside 2011-2023, '
      f'{bp["ticker"].nunique()} tickers')
print(bp.assign(date=bp['date'].dt.date, ret=bp['ret'].map(lambda x: f'{x:+.1%}')).to_string())
bp.to_csv(OUT_DIR/'survivorship_bad_prints.csv', index=False)
R_clean = R.mask(bad)                                    # drop ONLY that ticker-day return

win = (CAL >= A_START) & (CAL <= A_END)
dvol = pd.DataFrame(DVOL, index=['bars_2011', 'med_dvol_2011']).T
elig = dvol[(dvol['bars_2011'] >= 200) & (dvol['med_dvol_2011'] > 0)]
COHORT = elig.index[elig['med_dvol_2011'] <= elig['med_dvol_2011'].quantile(1/3)]
print(f'\n2011 dollar-volume cohort: {len(elig)} eligible (>=200 bars in 2011, volume > 0), '
      f'bottom tercile {len(COHORT)} tickers (median $vol <= {elig["med_dvol_2011"].quantile(1/3):,.0f})')

SER = {'EW_raw':   R.loc[win].mean(axis=1),
       'EW':       R_clean.loc[win].mean(axis=1),
       'EW_bot3':  R_clean.loc[win, COHORT].mean(axis=1)}
for tk in BENCH:
    SER[tk] = close_ret(BPX[tk], CAL).reindex(CAL[win])
S = pd.DataFrame(SER)
held = R_clean.loc[win].notna().sum(axis=1)
print(f'names held per day: min {held.min()}  median {int(held.median())}  max {held.max()}')
print('missing benchmark days in window:', S[BENCH].isna().sum().to_dict())

In [ ]:
# ===== Cell 5 — CAGR, max drawdown, annual gaps, verdict =====
N_YEARS = A_END.year - A_START.year + 1

def summarize(r):
    r = r.fillna(0.0); eq = (1 + r).cumprod()
    return {'CAGR': eq.iloc[-1] ** (1 / N_YEARS) - 1, 'MaxDD': float((eq / eq.cummax() - 1).min()),
            'total': eq.iloc[-1] - 1}

summ = pd.DataFrame({k: summarize(S[k]) for k in S}).T
print(f'2011-2023 ({N_YEARS} yrs), total return, daily series:')
print(summ.map(lambda x: f'{x:+.2%}').to_string())

ann = (1 + S.fillna(0.0)).groupby(S.index.year).prod() - 1
gaps = pd.DataFrame({f'EW-{b}': ann['EW'] - ann[b] for b in BENCH} |
                    {f'bot3-{b}': ann['EW_bot3'] - ann[b] for b in ['IJR', 'IWM']})
print('\nCalendar-year returns:')
print(ann.map(lambda x: f'{x:+.1%}').to_string())
print('\nAnnual gaps (EW = clean full universe; bot3 = bottom 2011 $vol tercile):')
print(gaps.map(lambda x: f'{x:+.1%}').to_string())
print('mean  ', ' '.join(f'{c}={v:+.2%}' for c, v in gaps.mean().items()))
print('>0 yrs', ' '.join(f'{c}={int((gaps[c] > 0).sum())}/{len(gaps)}' for c in gaps))

summ.to_csv(OUT_DIR/'survivorship_a2_summary.csv'); ann.to_csv(OUT_DIR/'survivorship_a2_annual.csv')
gaps.to_csv(OUT_DIR/'survivorship_a2_gaps.csv')

c = summ['CAGR']
tilt_full, tilt_b_ijr, tilt_b_iwm = c['EW'] - c['RSP'], c['EW_bot3'] - c['IJR'], c['EW_bot3'] - c['IWM']
size = lambda x: 'small' if abs(x) < 0.01 else ('moderate' if abs(x) < 0.03 else 'LARGE')
print(f'\nVERDICT: EW universe CAGR {c["EW"]:+.2%} vs RSP {c["RSP"]:+.2%} -> tilt {tilt_full:+.2%}/yr '
      f'({size(tilt_full)}; vs IWM {c["EW"] - c["IWM"]:+.2%}, vs SPY {c["EW"] - c["SPY"]:+.2%}); bottom tercile '
      f'{c["EW_bot3"]:+.2%} vs IJR {tilt_b_ijr:+.2%}/yr, vs IWM {tilt_b_iwm:+.2%}/yr ({size(tilt_b_iwm)}). '
      f'Lower bound (no delisting returns); includes daily-rebalance premium.')

In [ ]:
# ===== Cell 6 — equity curves (log) =====
import matplotlib.pyplot as plt
eq = (1 + S.fillna(0.0)).cumprod()
ax = eq[['EW', 'EW_bot3', 'RSP', 'IWM', 'IJR', 'SPY']].plot(logy=True, figsize=(11, 5), lw=1.2)
ax.set_title('Equal-weight corpus vs benchmarks, 2011-2023 (growth of $1, log)'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR/'survivorship_a2_equity.png', dpi=110); plt.show()